# Check Propofol PK/PD implementation

In [ ]:
%load_ext autoreload
%autoreload 2

import logging

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mstats


from propofol.dosing import Dosing
from propofol.haemo_pd import SuHaemoPD
from propofol.propofol_pkpd import EleveldPD, EleveldPK, PKPDSolver
from propofol.patient import EleveldPatient

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Eleveld 2018 PK/PD model

In [ ]:
patient = EleveldPatient(35, 70, 170, 'male', opiates=True)
p = PKPDSolver(patient, EleveldPK(patient), EleveldPD(patient))

p.print_model_parameters()
# (p.pk.V3(p.age, p.weight, p.bmi) / p.pk.V3(p.age_ref, p.weight_ref, p.bmi_ref))**0.75
print(f"k12/k21: {p.pk.k12 / p.pk.k21}")  # TODO: double check, but appears correct from comparing k10 to SimTIVA
print(f"k13/k31: {p.pk.k13 / p.pk.k31}")
print(f"Q2: {p.pk.Q2_arterial}")
print(f"Q3: {p.pk.Q3}")

In [ ]:
dose50 = p.find_dose_drug_effect_50()
t = np.linspace(0, 15, 15 * 60)
A1, A2, A3, Ce = p(t, [dose50, 0, 0, 0])

plt.plot(t, A1 / p.pk.V1, label='Cp (mcg / mL)')
plt.plot(t, Ce, label='Ce (mcg / mL)')
plt.plot(t, p.pd.Ce50 * np.ones(len(t)), 'k:', label='Ce50')
plt.xlabel('Time (minutes)')
plt.ylabel('Concentration (mcg / ml)')
plt.legend(loc=1)
plt.show()

bis = np.array([p.pd.bis(x) for x in Ce])
plt.plot(t, bis)
logger.info(f"Dose 50% drug effect: {dose50 / p.patient.weight:.2f} mg / kg")
logger.info(f"BIS min / BIS baseline: {bis.min() / p.pd.bis_baseline:.3f}")

#### Figure 5a
We recreate figure 5a of the paper

In [ ]:
age, weight, height = np.array([[0.003, 3.5, 50], [1, 10, 70], [3, 16, 95], [5, 18, 109], [10, 32, 139], [18, 70, 170], [35, 70, 170], [70, 70, 170]]).T
t = np.linspace(0, 15, 15 * 60)

for sex in ['male', 'female']:
    doses = []
    for a, w, h in zip(age, weight, height):

        patient = EleveldPatient(a, w, h, sex, opiates=True)
        p = PKPDSolver(patient, EleveldPK(patient), EleveldPD(patient))
        dose50 = p.find_dose_drug_effect_50()
        doses.append(dose50 / w)
        _, _, _, Ce = p(t, [dose50, 0, 0, 0])
        bis = np.array([p.pd.bis(x) for x in Ce])
        # logger.info(f"--- age: {a}, height: {h}, weight: {w} ---")
        # logger.info(f"Dose 50: {dose50:.3f}")
        # logger.info(f"BIS min: {bis.min()}; BIS baseline: {p.pd.bis_baseline:.3f}")
        # logger.info(f"BIS min / BIS baseline: {bis.min() / p.pd.bis_baseline:.3f}")
    plt.scatter(age, doses, label=sex)
    plt.fill_between([3, 16], [2.5, 2.5], [3.5, 3.5], color='green', alpha=0.4, zorder=-10)  # adults
    plt.fill_between([18, 55], [2, 2], [2.5, 2.5], color='green', alpha=0.4, zorder=-10)  # adults
    plt.fill_between([55, 70], [1, 1], [1.5, 1.5], color='green', alpha=0.4, zorder=-10)  # Elderly
    plt.legend(loc=1)
    plt.ylim(0, 4.7)
    plt.xlim(0, 90)
    plt.xlabel("Age (years)")
    plt.ylabel("Dose for 50% drug effect [mg / kg]")

### Individual variation
Simulation of BIS with between subject variability.

In [ ]:

bis_list = []
t = np.linspace(0, 15, 15 * 60)

# 10k samples takes about 11s
n_samples = 1000
for i in range(n_samples):
    if i == 0:

        pk_propofol = EleveldPK(patient, use_bsv=False)
        pd_propofol = EleveldPD(patient, use_bsv=False)
        p = PKPDSolver(patient, pk_propofol, pd_propofol)
        dose50 = p.find_dose_drug_effect_50()
    else:
        p.pk.draw_eta()
        p.pd.draw_eta()
    A1, A2, A3, Ce = p(t, [dose50, 0, 0, 0])
    bis = np.array([p.pd.bis(x) for x in Ce])
    bis_list.append(bis)
bis_list = np.array(bis_list)
bis_lo, bis_hi = mstats.mquantiles(bis_list, [0.025, 0.975], axis=0)
for ix, bis in enumerate(bis_list):
    if ix == 0:
        plt.plot(t, bis, 'r-', zorder=10, label=f"95% C.I.")
        plt.plot(t, bis_lo, 'r:', zorder=10)
        plt.plot(t, bis_hi, 'r:', zorder=10)
    else:
        plt.plot(t, bis, color='k', alpha=0.01)
plt.legend(loc=4)
plt.show()


## Su 2022 haemodynamic model

### Figure 2 of the paper
This figure assumes constant $C_p$ so we set all $k_{xy} = 0$ and $\frac{dA_0}{dt} = 0$ for the pk part of propofol (i.e. constant propofol concentration).

At $Cp = 0 \mu \text{g ml}^{-1}$ it should give $MAP = 86 \text{ mm Hg}$ and at  $Cp = 10 \mu \text{g ml}^{-1}$ it should give $MAP = 54 \text{mm Hg}$ according to the paper.
Heart rate changes from 56 bpm to 87 bmp and sv decreases from 82.9 to 75.6 ml (when $Cp = 0 \mu \text{g ml}^{-1}$ to $Cp = 1. \mu \text{g ml}^{-1}$) and increases to 91.2ml when $Cp = 10 \mu \text{g ml}^{-1}$.

*Note*: The $FB$ exponent in eqs. (6)-(8) of the paper misses a minus sign

In [ ]:
t = np.linspace(0, 180, 180*60+1)  # needs to be longer than one hour to properly converge
patient = EleveldPatient(35, 70, 170, 'male', opiates=True)
pd_propofol = propofol_pd=EleveldPD(patient)
pk_propofol = EleveldPK(patient)

# Set all pk parameters to 0
pk_propofol.k10 = 0
pk_propofol.k12 = 0
pk_propofol.k13 = 0
pk_propofol.k21 = 0
pk_propofol.k31 = 0
pk_propofol.ke0 = 0

pd_haemo = SuHaemoPD(patient=patient, pk_propofol= pk_propofol, pd_propofol=pd_propofol)

Cps = np.logspace(-1, 1, 21)
hrs = []
svs = []
MAPs = []
for Cp in Cps:
    y0 = [Cp * pd_haemo.pk_propofol.V1,  # fix A1 for constant Cp
          0, 
          0, 
          0,
        pd_haemo.base_sv, # base sv
        pd_haemo.base_hr,  # base hr
        pd_haemo.base_tpr, # base tpr
    pd_haemo.Theta12 * pd_haemo.base_hr]

    A1, A2, A3, Ce, sv, hr, MAP, tde = pd_haemo.solve_ode(t=t, 
                                                      y0 = y0,
                                                      dosing=None)
    hrs.append(hr[-1])
    svs.append(sv[-1])
    MAPs.append(MAP[-1])
logger.info(f"MAP at Cp={Cps[0]} mcg/ml (Cp={Cps[-1]} mcg/ml) = {MAPs[0]:.0f} mm Hg ({MAPs[-1]:.0f} mm Hg)")
logger.info(f"SV at Cp={Cps[0]} mcg/ml (Cp={Cps[-1]} mcg/ml) = {svs[0]:.1f} ml ({svs[-1]:.1f} ml)")
logger.info(f"SV min: {min(svs):.1f} ml")
logger.info(f"HR at Cp={Cps[0]} mcg/ml (Cp={Cps[-1]} mcg/ml) = {hrs[0]:.0f} min^-1 ({hrs[-1]:.0f} min^-1)")

# Plot
fig, ax1 = plt.subplots(figsize=(5, 6))
ax1.plot(Cps, svs, 'b-', label='SV [ml]')  # 'g-' means green solid line
ax1.plot(Cps, MAPs, 'k--', label='MAP [mm Hg]')  # 'g-' means green solid line
ax1.set_xlabel('Cp [mcg / ml]')
ax1.set_xscale('log')
ax1.set_ylabel('MAP (mm Hg) and SV (ml)')
ax1.set_ylim(53, 93)


ax2 = ax1.twinx()
ax2.plot(Cps, hrs, 'r:', label='HR [min^-1]')  # 'b-' means blue solid line
ax2.set_ylabel('HR [min^-1]')
ax2.set_ylim(53, 93)


Also simulate a realisting dosing scenario

In [ ]:
# dose50 = cm.pkpd_drug.find_dose_drug_effect_50()
t = np.linspace(0, 120, 120*60+1) 
patient = EleveldPatient(35, 70, 170, 'male', opiates=True)
pd_haemo = SuHaemoPD(patient=patient, pk_propofol= EleveldPK(patient), pd_propofol=EleveldPD(patient))
A1, A2, A3, Ce, sv, hr, MAP, tde = pd_haemo.solve_ode(t=t, 
                                                    y0 = None,
                                                    dosing=Dosing(patient=patient, strategy='default')
)
bis = np.array([pd_propofol.bis(x) for x in Ce])

# Store max Cp for later use
logger.info(f"max Cp =  {(A1 / pd_haemo.pk_propofol.V1).max():.2f}")
logger.info(f"max Ce =  {Ce.max():.2f}")
logger.info(f"min bis = {bis.min():.2f}")
logger.info(f"min MAP = {MAP.min():.2f}")

# Plot
plt.plot(t, A1 / pd_haemo.pk_propofol.V1, label='Cp (mcg / mL)')
plt.plot(t, Ce, label='Ce (mcg / mL)')
# plt.plot(t, cm.pkpd_drug.pd.Ce50 * np.ones(len(t)), 'k:', label='Ce50')
plt.xlabel('Time (minutes)')
plt.ylabel('Concentration (mcg / ml)')
plt.legend(loc=1)
plt.show()

plt.figure()
plt.plot(t, bis)
plt.title('BIS')
# logger.info(f"Dose 50% drug effect: {dose50 / pd_haemo.patient.weight:.2f} mg / kg")

fig, ax1 = plt.subplots()
ax1.plot(t, sv, 'b-', label='SV [ml]')  # 'g-' means green solid line
ax1.plot(t, MAP, 'k--', label='MAP [mm Hg]')  # 'g-' means green solid line
ax1.set_xlabel('time [min]')
ax1.set_xlim(0, 125)
ax1.set_ylabel('MAP (mm Hg) and SV (ml)')
ax1.set_ylim(53, 93)


ax2 = ax1.twinx()
ax2.plot(t, hr, 'r:', label='HR [min^-1]')  # 'b-' means blue solid line
ax2.set_ylabel('HR [min^-1]')
ax2.set_ylim(53, 93)



Now simulate the same scenario, but instead of using the exact dosing strategy use a more coarse-grained 1min cumulative dose.

*Note*: compared to more fine grained dosing information (during bolus or titration)
* Max $C_p$ lower
* Max $C_e$ higher (delayed effect)
* $bis$ and $map$ similar

In [ ]:
d = Dosing(patient=patient, strategy='default')

t = np.linspace(0, 120, 120*60+1)
patient = EleveldPatient(35, 70, 170, 'male', opiates=True)
pd_haemo = SuHaemoPD(patient=patient, pk_propofol= EleveldPK(patient), pd_propofol=EleveldPD(patient))
A1, A2, A3, Ce, sv, hr, MAP, tde = pd_haemo.solve_ode(t=t, 
                                                    y0 = None,
                                                    dosing=Dosing(patient=patient, 
                                                                  cumulative_dose=d.get_cumulative_dose_from_strategy())
)
bis = np.array([pd_propofol.bis(x) for x in Ce])

logger.info(f"Max Cp (from cum. dose) =  {(A1 / pd_haemo.pk_propofol.V1).max():.2f} (mcg / mL)")
logger.info(f"Max Ce (from cum. dose) =  {Ce.max():.2f} (mcg / mL)")
logger.info(f"min bis (from cum. dose) = {bis.min():.2f}")
logger.info(f"min MAP (from cum. dose) = {MAP.min():.2f}")